In [19]:
import matplotlib.pyplot as plt
import nelpy as nel
import nelpy.plotting as npl
import numpy as np
import seaborn as sns
from neuro_py.io import loading
from neuro_py.plotting.figure_helpers import set_size
from neuro_py.tuning import maps
from skimage import measure
import os
import pandas as pd
import scipy.io as sio
import logging

logging.getLogger().setLevel(logging.ERROR)
basepath =  r"W:\data\PPP\PPP7\day8"
basename=os.path.basename(basepath)# Define the new folder path
#create working folder
new_folder_path = os.path.join(basepath, "complex_system")
brain_region="CA1"
animal="mouse"   #or Mouse
cell_type='all'
path_parts = basepath.split("\\")  # Split the path using backslash
animal_name = path_parts[-2]  # Second-to-last part is the animal name
day = path_parts[-1]  # Last part is the day
save_dir = r"D:\Documents\GitHub\R_Project\PPP\codebook\Criticality_analysis\complex_system\manifold"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
# Create the folder if it doesn't exist
if not os.path.exists(new_folder_path):
    os.makedirs(new_folder_path)
    print(f"Folder created: {new_folder_path}")
else:
    print(f"Folder already exists: {new_folder_path}")
# load in spike data from hpc pyramidal cells
st, cm = loading.load_spikes(
    basepath, brainRegion=brain_region                          # putativeCellType="Pyr"
)
# load theta epochs
state_dict = loading.load_SleepState_states(basepath)
nrem_epochs = nel.EpochArray(
    state_dict["NREMstate"],
)
awake_epochs = nel.EpochArray(
    state_dict["WAKEstate"],
)
# theta_epochs = nel.EpochArray(
#     state_dict["THETA"],
# )
rem_epochs=nel.EpochArray(state_dict["REMstate"],)
try:
    # Try default loader
    epoch_df = loading.load_epoch(basepath)
except Exception as e:
    print(f"load_epoch failed with error: {e}")
    print("Falling back to manual .mat file load...")

    # Load the .mat file
    mat = sio.loadmat(os.path.join(basepath, "DayNight25.MergePoints.events.mat"), squeeze_me=True)

    # Extract the 'MergePoints' structure
    merge_points = mat['MergePoints']

    # Extract the timestamps array: shape (N, 2)
    timestamps = merge_points['timestamps']
    # Unwrap the array inside
    timestamps = np.array(timestamps.tolist())

    epoch_df = pd.DataFrame(timestamps, columns=["startTime", "stopTime"])

sleep_epochs = nel.EpochArray(np.array([epoch_df.startTime, epoch_df.stopTime]).T)
E_spikes = st.time[(cm.putativeCellType == 'Pyramidal Cell')]
I_spikes= st.time[(cm.putativeCellType.str.contains("Interneuron"))]
spike_times=st.time
n_cell=len(spike_times)
print(f"number of cells: {n_cell}")

Folder already exists: W:\data\PPP\PPP7\day8\complex_system
number of cells: 303


In [20]:
import os, re, numpy as np, pandas as pd
import glob
def _load_mat_any(mat_path):
    from scipy.io import loadmat
    try:
        return loadmat(
            mat_path,
            squeeze_me=True,
            struct_as_record=False,
            simplify_cells=True
        )
    except TypeError:
        return loadmat(
            mat_path,
            squeeze_me=True,
            struct_as_record=False
        )
    except NotImplementedError:
        import mat73
        return mat73.loadmat(mat_path)
def _to_1d_array(x):
    """
    Robustly coerce MATLAB-loaded field to a 1D numpy array.
    - Scalars -> 1-length array
    - Lists/tuples/ndarrays -> flattened 1D
    - None / missing -> empty array
    """
    if x is None:
        return np.array([], dtype=float)
    arr = np.asarray(x, dtype=object)  # start as object to avoid surprises
    if arr.ndim == 0:
        # scalar or unsized object
        return np.atleast_1d(arr).astype(float, copy=False)
    # flatten and try float cast (ignore non-numeric gracefully)
    try:
        return np.asarray(arr).ravel().astype(float, copy=False)
    except (ValueError, TypeError):
        # fallback: best-effort, convert elementwise
        return np.array([float(v) for v in np.ravel(arr)], dtype=float)

def _to_state_str(s):
    # MATLAB char arrays sometimes come in as numpy arrays
    try:
        if isinstance(s, (bytes, bytearray)):
            return s.decode('utf-8', 'ignore')
        if isinstance(s, str):
            return s
        # arrays or weird containers -> join characters
        arr = np.asarray(s)
        if arr.ndim == 0:
            return str(arr.item())
        # e.g., array of characters
        return ''.join(arr.tolist()) if arr.dtype.kind in ('U','S') else str(arr)
    except Exception:
        return str(s)

def load_d2_timeline_complete(basepath, region):
    """Complete version that loads ALL sleep-state data, including d2_err."""
    region = region.upper()
    search_dir = os.path.join(basepath, "complex_system")
    # pats = [f"*_{region}_D2_time_resolved_rippleablate.mat", f"*_{region}_D2_time_rippleablate.mat"]
    pats = [f"*_{region}_D2_time_resolved_error_bar.mat", f"*_{region}_D2_time_resolved_error_bar.mat"]
    files = []
    for pat in pats:
        files.extend(glob.glob(os.path.join(search_dir, pat)))
    if not files:
        raise FileNotFoundError(f"No *_{region}_D2_time*.mat under {search_dir}")

    mat_path = max(files, key=os.path.getmtime)
    print(f"Loading: {os.path.basename(mat_path)}")

    mat = _load_mat_any(mat_path)
    results = mat['results']
    params = mat.get('params', {})

    # Make results iterable no matter if it's a list, ndarray, or dict
    if isinstance(results, dict):
        iter_results = list(results.values())
    else:
        iter_results = list(results if isinstance(results, (list, tuple)) else np.atleast_1d(results))

    print(f"📊 Found {len(iter_results)} result items (sleep states)")
    print(f"📊 Parameters: {params}")

    # Try to infer animal/day from filename if available
    fname = os.path.basename(mat_path)
    animal_name = None
    day = None
    m_day = re.search(r'day(\d+)', fname, flags=re.I)
    if m_day:
        day = int(m_day.group(1))
    m_animal = re.match(r'([^_]+)_day\d+_', fname)
    if m_animal:
        animal_name = m_animal.group(1)

    all_centers, all_d2, all_d2err, all_states = [], [], [], []
    n_states_with_err = 0

    for i, result_item in enumerate(iter_results):
        # Access as dict-like regardless of MATLAB struct quirks
        state = _to_state_str(result_item.get('state', 'UNKNOWN')) if hasattr(result_item, 'get') else \
                _to_state_str(result_item['state']) if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) else 'UNKNOWN'

        centers_raw = result_item.get('center', []) if hasattr(result_item, 'get') else \
                      (result_item['center'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'center' in result_item else [])
        d2_raw = result_item.get('d2', []) if hasattr(result_item, 'get') else \
                 (result_item['d2'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'd2' in result_item else [])
        d2err_raw = result_item.get('d2_err', []) if hasattr(result_item, 'get') else \
                    (result_item['d2_err'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'd2_err' in result_item else [])

        centers  = _to_1d_array(centers_raw)
        d2_vals  = _to_1d_array(d2_raw)
        d2_err   = _to_1d_array(d2err_raw)

        # Presence flag (some states may not have d2_err)
        has_err = d2_err.size > 0 and np.any(np.isfinite(d2_err))
        n_states_with_err += int(has_err)

        # Harmonize lengths: truncate all to the common min
        sizes = [centers.size, d2_vals.size] + ([d2_err.size] if has_err else [])
        n_pts = min(sizes) if min(sizes) > 0 else 0
        print(f"   State {i}: {state} - requested points={centers.size}, actual used={n_pts}")

        if n_pts == 0:
            print("     ⚠️ Empty state - no data points")
            continue

        centers = centers[:n_pts]
        d2_vals = d2_vals[:n_pts]
        d2_err  = (d2_err[:n_pts] if has_err else np.full(n_pts, np.nan, dtype=float))

        # Ranges
        try:
            print(f"     Time range: {float(np.nanmin(centers))} to {float(np.nanmax(centers))}")
        except ValueError:
            print("     Time range: (all NaN)")

        try:
            print(f"     D2 range: {float(np.nanmin(d2_vals)):.4f} to {float(np.nanmax(d2_vals)):.4f}")
        except ValueError:
            print("     D2 range: (all NaN)")

        if np.any(np.isfinite(d2_err)):
            try:
                dmin, dmax = float(np.nanmin(d2_err)), float(np.nanmax(d2_err))
                print(f"     D2_ERR range: {dmin:.4f} to {dmax:.4f}")
            except ValueError:
                print("     D2_ERR range: (all NaN)")
        else:
            print("     D2_ERR: missing → filled with NaN")

        all_centers.append(centers)
        all_d2.append(d2_vals)
        all_d2err.append(d2_err)
        all_states.append(np.array([state]*n_pts, dtype=object))

    if len(all_centers) == 0:
        raise ValueError("No data found in any sleep state - all states are empty")

    # Concatenate
    all_centers = np.concatenate(all_centers)
    all_d2      = np.concatenate(all_d2)
    all_d2err   = np.concatenate(all_d2err)
    all_states  = np.concatenate(all_states)

    # DataFrame (include d2_err)
    df = pd.DataFrame({
        'center_s': all_centers.astype(float, copy=False),
        'd2':       all_d2.astype(float, copy=False),
        'd2_err':   all_d2err.astype(float, copy=False),
        'state':    all_states.astype(str)
    }).sort_values('center_s').reset_index(drop=True)

    print(f"\n✅ Complete timeline created:")
    print(f"📊 Total data points: {len(df)}")
    print(f"📊 Time range: {df['center_s'].min()} to {df['center_s'].max()}")
    print(f"📊 Duration: {(df['center_s'].max() - df['center_s'].min()):.1f} seconds")
    print(f"📊 State distribution:")
    print(df['state'].value_counts())

    # Empty-state tracker
    empty_states = []
    for ri in iter_results:
        st = _to_state_str(ri.get('state','UNKNOWN')) if hasattr(ri,'get') else \
             _to_state_str(ri['state']) if (isinstance(ri,dict) or hasattr(ri,'__getitem__')) and 'state' in ri else 'UNKNOWN'
        csz = np.size(np.atleast_1d(ri.get('center', []))) if hasattr(ri,'get') else \
              np.size(np.atleast_1d(ri['center'])) if (isinstance(ri,dict) or hasattr(ri,'__getitem__')) and 'center' in ri else 0
        if csz == 0:
            empty_states.append(st)

    meta = {
        "animal_name": animal_name,
        "day": day,
        "region": region,
        "path": mat_path,
        "params": params,
        "columns": list(df.columns),
        "shape": df.shape,
        "n_states": len(iter_results),
        "states_with_d2_err": n_states_with_err,
        "empty_states": empty_states
    }

    return df, meta



# Load the CA1 d2 timeline
try:
    ca1_complete_df, ca1_complete_meta = load_d2_timeline_complete(basepath, "CA1")
    
    # pfc_complete_df, pfc_complete_meta = load_d2_timeline_complete(basepath, "PFC")

    
except Exception as e:
    print(f"❌ Failed: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*60)

Loading: day8_CA1_D2_time_resolved_error_bar.mat
📊 Found 3 result items (sleep states)
📊 Parameters: {'window_size': 30, 'step_size': 2, 'guard_sec': 5, 'deltaT': 0.04, 'order': 20, 'quant': 0.1, 'brain_region': 'CA1', 'n_cells': 303, 'total_spikes': 11092851}
   State 0: NREM - requested points=3677, actual used=3677
     Time range: 21.0 to 25578.0
     D2 range: 0.0267 to 0.2224
     D2_ERR range: 0.0102 to 0.0322
   State 1: REM - requested points=66, actual used=66
     Time range: 5135.0 to 25666.0
     D2 range: 0.0571 to 0.1233
     D2_ERR range: 0.0151 to 0.0221
   State 2: WAKE - requested points=5474, actual used=5474
     Time range: 1867.0 to 25190.0
     D2 range: 0.0178 to 0.2702
     D2_ERR range: 0.0078 to 0.0347

✅ Complete timeline created:
📊 Total data points: 9217
📊 Time range: 21.0 to 25666.0
📊 Duration: 25645.0 seconds
📊 State distribution:
state
WAKE    5474
NREM    3677
REM       66
Name: count, dtype: int64



In [21]:
epoch_df

,name,startTime,stopTime,environment,behavioralParadigm,notes,manipulation,stimuli,basepath
0,presleep_230427_093311,0.0000,6643.12315,sleep,NaN,NaN,NaN,NaN,W:\data\PPP\PPP7\day8
1,T_long_230427_113208,6643.1232,9342.18235,T_maze,1,NaN,NaN,NaN,W:\data\PPP\PPP7\day8
2,midsleep_230427_122344,9342.1824,15837.94555,sleep,NaN,NaN,NaN,NaN,W:\data\PPP\PPP7\day8
3,T_long2_230427_141812,15837.9456,19580.24315,T_maze,1,77% out of 46 trials in free T_maze will keep ...,NaN,NaN,W:\data\PPP\PPP7\day8
4,postsleep_230427_153045,19580.2432,25720.51835,sleep,NaN,NaN,NaN,NaN,W:\data\PPP\PPP7\day8


In [22]:
import numpy as np
from scipy.signal import welch
import nelpy as nel

def _flatten_spike_times(spike_times):
    """Accepts list/tuple/array of arrays or a single 1D array, returns sorted 1D float array."""
    if isinstance(spike_times, (list, tuple)):
        # list of arrays or scalars
        if len(spike_times) == 0:
            return np.array([], dtype=float)
        if hasattr(spike_times[0], "__len__") and not np.isscalar(spike_times[0]):
            times = np.concatenate([np.asarray(st, dtype=float).ravel()
                                    for st in spike_times if st is not None and len(st) > 0])
        else:
            times = np.asarray(spike_times, dtype=float).ravel()
    else:
        arr = np.asarray(spike_times, dtype=object)
        if arr.dtype == object:
            times = np.concatenate([np.asarray(st, dtype=float).ravel()
                                    for st in arr if st is not None and len(st) > 0])
        else:
            times = arr.astype(float).ravel()

    times = times[np.isfinite(times)]
    if times.size == 0:
        raise ValueError("No valid spike times found after flattening.")
    return np.sort(times)

def bin_spikes(spike_times, bin_size=0.04, t_start=None, t_stop=None):
    st = _flatten_spike_times(spike_times)
    if t_start is None: t_start = float(np.floor(st.min()))
    if t_stop  is None: t_stop  = float(np.ceil(st.max()))
    edges = np.arange(t_start, t_stop + bin_size, bin_size)
    counts, _ = np.histogram(st, bins=edges)
    rates = counts / bin_size
    t = edges[:-1] + bin_size/2
    return t, rates

# def segments_from_epochs(t, x, epochs):
#     arr = epochs.data if isinstance(epochs, nel.EpochArray) else np.asarray(epochs)
#     return [x[(t >= s) & (t < e)] for s, e in arr if np.any((t >= s) & (t < e))]

In [23]:
# Create plot for random 200 second segment containing all states
import matplotlib.pyplot as plt
import numpy as np
import random

# First compute MUA from spike times
print("Computing MUA firing rate from spike times...")
print("="*50)

# Compute MUA firing rate with 0.04 second bins
time_bins, firing_rate = bin_spikes(spike_times, bin_size=0.04)

Computing MUA firing rate from spike times...


In [24]:
import numpy as np
import pandas as pd

# --- Inputs assumed to exist ---
# spike_times
# time_bins, firing_rate  # from bin_spikes(..., bin_size=0.04)
# ca1_complete_df          # has a 'center_s' column

# 1) Make sure we have bin *centers* that match firing_rate length
tb = np.asarray(time_bins, float).ravel()
fr = np.asarray(firing_rate, float).ravel()

if tb.size == fr.size + 1:            # looks like edges → make centers
    t = 0.5 * (tb[:-1] + tb[1:])
elif tb.size == fr.size:               # already centers
    t = tb
else:
    raise ValueError("time_bins length must equal firing_rate length or be exactly one longer (edges).")

# 2) Clean + sort
ok = np.isfinite(t) & np.isfinite(fr)
t, fr = t[ok], fr[ok]
ordr = np.argsort(t)
t, fr = t[ordr], fr[ordr]

# 3) 30 s centered moving average of MUA (Hz)
dt = float(np.median(np.diff(t)))
win_s = 30.0
win_n = max(int(round(win_s / dt)), 1)

# Use pandas rolling for a centered window; allow some tolerance near edges
fr_series = pd.Series(fr, index=pd.Index(t, name="t"))
fr_30s = fr_series.rolling(window=win_n, center=True, min_periods=max(1, win_n//4)).mean().to_numpy()

# 4) Interpolate the smoothed MUA onto the D² centers
centers = np.asarray(ca1_complete_df["center_s"], float)
mua_at_centers = np.interp(centers, t, fr_30s, left=np.nan, right=np.nan)

# 5) Save to dataframe (Hz)
ca1_complete_df["MUA"] = mua_at_centers


SWR and barrage

In [25]:
import numpy as np
import scipy.io as sio      
import scipy
# Load ripple events
ripple_events = loading.load_ripples_events(basepath=basepath)
ripple_epochs = nel.EpochArray(np.array([ripple_events.start, ripple_events.stop]).T)
ripple_peaks=ripple_events.peaks
    # Load barrage events
ripple_epochs=nrem_epochs&ripple_epochs
try:
    # Load barrage events
    #brpath=r"W:\data\PPP\CA2\PPP7\day8"
    barrage_events = loading.load_barrage_events(basepath=basepath)  
    # Create barrage epochs
    barrage_epochs = nel.EpochArray(np.array([barrage_events.start, barrage_events.stop]).T)
    # Find barrages in NREM epochs
    barrages_in_nrem = barrage_epochs & nrem_epochs
    mat_path = os.path.join(basepath, f'{basename}.HSEn2.events.mat')

    data = scipy.io.loadmat(mat_path, struct_as_record=False, squeeze_me=True)
    barrage = data['HSEn2']
    barrage_timestamps= barrage.timestamps 
    barrage_epochs=nrem_epochs&barrage_epochs
except Exception as e:
    # General exception for unexpected errors
    print(f"No barrage file: {e}")
    barrage_epochs=[]
    # Extract start times

No barrage file: object of too small depth for desired array


d2 pre post

In [26]:
import numpy as np
import pandas as pd
from typing import Optional

def annotate_d2_with_sleep_and_ripple_rate(
    ca1_complete_df: pd.DataFrame,
    epoch_df: pd.DataFrame,
    ripple_epochs_data: np.ndarray,
    window_len: float =30,   # seconds
) -> pd.DataFrame:
    """
    - Adds 'sleep_epoch' to ca1_complete_df: {'presleep','midsleep','postsleep', np.nan}
    - Adds 'ripple_rate_30s' to ca1_complete_df: events/sec within +/- window_len/2 around center_s.
    
    Assumptions:
      * epoch_df has columns: ['name','startTime','stopTime','environment']
      * Only epochs with environment=='sleep' are used for labeling.
      * Ripple events provided as Nx2 array [start, end] in seconds.
        For point events, pass Nx2 with end=start.
    """
    df = ca1_complete_df.copy()

    # -------- 1) Build sleep interval table with labels
    sleep_epochs = epoch_df.copy()
    sleep_epochs = sleep_epochs[sleep_epochs['environment'].astype(str).str.lower()=='sleep'].copy()

    # Normalize names to lower and assign coarse label
    def _label_from_name(n: str) -> Optional[str]:
        s = str(n).lower()
        if 'presleep' in s or 'pre_sleep' in s or 'pre' in s and 'sleep' in s:
            return 'presleep'
        if 'midsleep' in s or 'mid_sleep' in s:
            return 'midsleep'
        if 'postsleep' in s or 'post_sleep' in s or ('post' in s and 'sleep' in s):
            return 'postsleep'
        return None

    sleep_epochs['sleep_epoch'] = sleep_epochs['name'].map(_label_from_name)
    sleep_epochs = sleep_epochs.dropna(subset=['sleep_epoch'])
    # Keep only necessary columns and ensure numeric
    sleep_epochs = sleep_epochs[['startTime','stopTime','sleep_epoch']].astype({'startTime':float,'stopTime':float})

    # -------- Map each center_s to an interval label
    # We'll do this by merging after expanding intervals to non-overlapping rows per center
    centers = df[['center_s']].astype(float).copy()
    centers['row_id___'] = np.arange(len(centers))

    # For efficient interval lookup, sort and then use merge_asof on both boundaries.
    starts = sleep_epochs[['startTime','sleep_epoch']].rename(columns={'startTime':'bound'})
    starts['flag'] = 1
    stops  = sleep_epochs[['stopTime','sleep_epoch']].rename(columns={'stopTime':'bound'})
    stops['flag'] = -1
    bounds = pd.concat([starts, stops], ignore_index=True).sort_values('bound')

    # Sweep-line to know which interval each center falls into
    centers_sorted = centers.sort_values('center_s')
    idx_b = np.searchsorted(bounds['bound'].to_numpy(), centers_sorted['center_s'].to_numpy(), side='right') - 1
    # cumulative sum per label is tricky; simpler: vectorized membership by intervals
    # Use pandas.IntervalIndex for clarity:
    iv = pd.IntervalIndex.from_arrays(sleep_epochs['startTime'].to_numpy(),
                                      sleep_epochs['stopTime'].to_numpy(),
                                      closed='both')
    # get label for each center via interval containment
    membership_codes = iv.get_indexer(centers_sorted['center_s'].to_numpy())
    labels = pd.Series(np.where(membership_codes>=0,
                                sleep_epochs['sleep_epoch'].to_numpy()[membership_codes],
                                np.nan),
                       index=centers_sorted.index)
    centers_sorted['sleep_epoch'] = labels
    # restore original order
    centers = centers_sorted.sort_values('row_id___').drop(columns=['row_id___'])

    df['sleep_epoch'] = centers['sleep_epoch'].values

    # -------- 2) Ripple rate in ±window_len/2 around each center_s
    half = window_len / 2.0
    # ripple_epochs_data expected shape (N,2): start,end.
    rip = np.asarray(ripple_epochs_data, dtype=float)
    if rip.ndim == 1:
        # If only starts are provided, treat end=start (instantaneous)
        rip = np.column_stack([rip, rip])
    elif rip.shape[1] != 2:
        raise ValueError("ripple_epochs_data must be (N,2) [start,end] in seconds.")

    r_start = rip[:,0]
    r_end   = rip[:,1]

    centers_arr = df['center_s'].to_numpy(dtype=float)
    win_starts = centers_arr - half
    win_ends   = centers_arr + half

    # Count overlap: (r_start < win_end) & (r_end > win_start)
    # Vectorized via broadcasting in manageable chunks to avoid huge memory use
    counts = np.zeros(len(centers_arr), dtype=int)
    chunk = 5000  # adjust if needed
    for i0 in range(0, len(centers_arr), chunk):
        i1 = i0 + chunk
        ws = win_starts[i0:i1][:, None]
        we = win_ends[i0:i1][:, None]
        overlap = (r_start[None, :] < we) & (r_end[None, :] > ws)
        counts[i0:i1] = overlap.sum(axis=1)

    df['ripple_rate_30s'] = counts / window_len  # events per second
    # (events per minute: multiply by 60)

    return df

# ---------------- Example usage ----------------
# ca1_complete_df = ...  # has columns ['center_s','d2','state']
# epoch_df = ...         # has ['name','startTime','stopTime','environment']
# ripple_epochs_data = ...  # Nx2 array of [start,end] in seconds

ca1_complete_df = annotate_d2_with_sleep_and_ripple_rate(ca1_complete_df, epoch_df, ripple_epochs.data)
ca1_complete_df.head()

,center_s,d2,d2_err,state,MUA,sleep_epoch,ripple_rate_30s
0,21.0,0.056276,0.014307,NREM,531.833333,presleep,0.600000
1,23.0,0.054851,0.013983,NREM,530.000000,presleep,0.600000
2,25.0,0.052163,0.013785,NREM,523.350000,presleep,0.566667
3,27.0,0.076646,0.016445,NREM,503.266667,presleep,0.566667
4,29.0,0.083136,0.017082,NREM,495.650000,presleep,0.533333


In [27]:

from pathlib import Path
import pandas as pd

# # Inputs:
# ca1_complete_df : pandas DataFrame
# animal_name     : str
# day             : int | str
# # brain_region    : str

out_dir = Path(r"D:\Documents\GitHub\Sleep_criticaltiy\data\ripple_barr\manipulation\pre_post")
out_dir.mkdir(parents=True, exist_ok=True)

fname = f"{animal_name}_{day}_{brain_region}.pkl"
save_path = out_dir / fname

ca1_complete_df.to_pickle(save_path)
print(f"Saved to: {save_path}")




Saved to: D:\Documents\GitHub\Sleep_criticaltiy\data\ripple_barr\manipulation\pre_post\PPP7_day8_CA1.pkl
